In [7]:
%pip install -q numpy scipy pandas matplotlib seaborn scikit-learn torch torchvision

Note: you may need to restart the kernel to use updated packages.


In [9]:
import scipy
import numpy as np # manipuler des tableaux à N dimensions
import pandas as pd # tableaux de données (data frame)
import matplotlib.pyplot as plt # visualisation de données
import seaborn # visualisation de données avancée
from sklearn import preprocessing # modèles de ML de base

import os
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms

In [10]:
device = "cuda:0" if torch.cuda.is_available() else "cpu" # 0 indique le GPU que vous allez utiliser
print(f"Using {device} device")

Using cpu device


In [11]:
from torchvision import datasets
from torchvision import transforms

transform = transforms.Compose([   # combine plusieurs transformations en un seul pipeline
    transforms.ToTensor(),  # convertit l'image en tenseur
    transforms.Normalize((0.5,), (0.5,))   # normalise les valeurs des pixels avec mean = 0.5 et std = 0.5 ; le nouveau x vaut donc (x - mean)/std
])

traindata = datasets.MNIST(
    root="~/data",train=True,download=True,
    transform=transform
)

testdata = datasets.MNIST(
    root="~/data",train=False,download=True,
    transform=transform
)


100%|██████████| 9.91M/9.91M [00:00<00:00, 60.6MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.68MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 14.9MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 6.29MB/s]


In [12]:
traindata

Dataset MNIST
    Number of datapoints: 60000
    Root location: /root/data
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=(0.5,), std=(0.5,))
           )

In [14]:
import numpy as np
from torch.utils.data import Subset, DataLoader

# 1. Récupérer le tableau des étiquettes depuis traindata
targets = np.array(traindata.targets)
labeled_indices = []

# 2. Choisir aléatoirement 10 échantillons pour chaque chiffre de 0 à 9 (100 échantillons au total)
num_labeled_per_class = 10
np.random.seed(42) # Fixer la graine pour obtenir des résultats identiques à chaque exécution

for digit in range(10):
    indices_of_digit = np.where(targets == digit)[0]
    selected = np.random.choice(indices_of_digit, num_labeled_per_class, replace=False)
    labeled_indices.extend(selected)

# 3. Le reste de l'ensemble d'entraînement constitue l'ensemble non étiqueté (Unlabeled)
all_indices = np.arange(len(traindata))
unlabeled_indices = np.setdiff1d(all_indices, labeled_indices)

# 4. Créer les Subset à partir de traindata
labeled_subset = Subset(traindata, labeled_indices)
unlabeled_subset = Subset(traindata, unlabeled_indices)

# 5. Encapsuler dans des DataLoader pour alimenter le modèle
batch_size = 64
labeled_loader = DataLoader(labeled_subset, batch_size=batch_size, shuffle=True)
unlabeled_loader = DataLoader(unlabeled_subset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(testdata, batch_size=batch_size, shuffle=False)

print(f"Séparation des données réussie !")
print(f"- Nombre d'échantillons étiquetés (fine-tuning) : {len(labeled_subset)}")
print(f"- Nombre d'échantillons non étiquetés (pré-entraînement auto-supervisé) : {len(unlabeled_subset)}")
print(f"- Nombre d'échantillons de test : {len(testdata)}")

Séparation des données réussie !
- Nombre d'échantillons étiquetés (fine-tuning) : 100
- Nombre d'échantillons non étiquetés (pré-entraînement auto-supervisé) : 59900
- Nombre d'échantillons de test : 10000


/tmp/ipykernel_50/2519852593.py:5: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments.
  targets = np.array(traindata.targets)


In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class CNNEncoder(nn.Module):
    def __init__(self, feature_dim=128):
        super(CNNEncoder, self).__init__()
        
        # Partie d'extraction des caractéristiques (backbone CNN)
        # Input: [Batch_size, 1, 28, 28]
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1) # -> [B, 32, 28, 28]
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1) # -> [B, 64, 28, 28]
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)                  # Réduit la taille de moitié : 28x28 -> 14x14 -> 7x7
        
        # Partie linéaire : compresse les caractéristiques en un vecteur de dimension feature_dim (128 par défaut)
        self.fc1 = nn.Linear(64 * 7 * 7, 256)
        self.fc2 = nn.Linear(256, feature_dim)

    def forward(self, x):
        # Bloc de convolution 1
        x = self.pool(F.relu(self.conv1(x))) # La taille devient 14x14
        # Bloc de convolution 2
        x = self.pool(F.relu(self.conv2(x))) # La taille devient 7x7
        
        # Aplatissement pour l'entrée de la couche Fully Connected
        x = x.view(x.size(0), -1) 
        x = F.relu(self.fc1(x))
        
        # Dernière couche de projection pour produire le vecteur de caractéristiques z
        z = self.fc2(x)
        
        # La normalisation L2 est essentielle en SSL : elle ramène le vecteur à une longueur de 1
        z = F.normalize(z, p=2, dim=1)
        return z

# Initialiser le réseau f et le tester avec un batch d'images quelconque issu de unlabeled_loader
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
f_encoder = CNNEncoder(feature_dim=128).to(device)

print(f"Réseau encodeur f initialisé avec succès sur l'appareil : {device}")
print(f_encoder)

# Tester un batch dans le réseau pour vérifier que la forme du vecteur de sortie est correcte
sample_images, _ = next(iter(unlabeled_loader))
sample_images = sample_images.to(device)
sample_z = f_encoder(sample_images)
print(f"Dimension du vecteur de caractéristiques en sortie (Batch_size, Feature_dim) : {sample_z.shape}")

Réseau encodeur f initialisé avec succès sur l'appareil : cpu
CNNEncoder(
  (conv1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=3136, out_features=256, bias=True)
  (fc2): Linear(in_features=256, out_features=128, bias=True)
)


NameError: name 'unlabeled_loader' is not defined

In [16]:
import copy

# 1. Créer le réseau g avec la même architecture que f et l'envoyer sur l'appareil (GPU/CPU)
g_encoder = CNNEncoder(feature_dim=128).to(device)

# 2. Copier à l'identique tous les poids (weights) de f vers g à l'instant initial
g_encoder.load_state_dict(f_encoder.state_dict())

# 3. Désactiver complètement le calcul du gradient pour g (g n'apprend pas par rétropropagation)
for param in g_encoder.parameters():
    param.requires_grad = False

print("Réseau g (encodeur cible) initialisé avec succès !")

# 4. Écrire la fonction de mise à jour des poids de g selon la formule EMA après chaque étape d'apprentissage
def update_momentum_encoder(f_net, g_net, m=0.999):
    """
    Met à jour les poids de g à partir de ceux de f :
    theta_g = m * theta_g + (1 - m) * theta_f
    (m proche de 1, par exemple 0.999, permet à g d'évoluer très lentement et de manière stable)
    """
    with torch.no_grad():
        for param_f, param_g in zip(f_net.parameters(), g_net.parameters()):
            param_g.data.mul_(m).add_(param_f.data, alpha=1 - m)

print("Fonction de mise à jour EMA pour g définie avec succès !")

Réseau g (encodeur cible) initialisé avec succès !
Fonction de mise à jour EMA pour g définie avec succès !


In [17]:
class MemoryBank(nn.Module):
    def __init__(self, feature_dim=128, memory_size=4096):
        super(MemoryBank, self).__init__()
        self.feature_dim = feature_dim
        self.memory_size = memory_size
        
        # Initialiser une banque de mémoire sous forme de buffer PyTorch (ce n'est pas un poids appris par rétropropagation)
        # Au départ, remplie de valeurs aléatoires avec normalisation L2
        init_memory = torch.randn(memory_size, feature_dim)
        init_memory = F.normalize(init_memory, dim=1)
        self.register_buffer("memory", init_memory)
        
        # Pointeur indiquant la prochaine position à écraser dans la file
        self.register_buffer("ptr", torch.zeros(1, dtype=torch.long))

    @torch.no_grad()
    def update(self, z_keys):
        """
        Met à jour la banque de mémoire avec les nouveaux vecteurs de caractéristiques (z_keys issus du réseau g)
        selon un mécanisme de file circulaire (FIFO - First In, First Out).
        """
        batch_size = z_keys.shape[0]
        ptr = int(self.ptr)
        
        # Si la taille du batch dépasse l'espace restant dans la mémoire, on tronque
        if ptr + batch_size > self.memory_size:
            # Ne garder que ce qu'il faut pour remplir jusqu'à la fin de la banque
            batch_size = self.memory_size - ptr
            z_keys = z_keys[:batch_size]
        
        # Écraser avec les nouveaux vecteurs à la position courante du pointeur
        self.memory[ptr:ptr + batch_size, :] = z_keys.detach()
        
        # Déplacer le pointeur ; en cas de dépassement de la fin, revenir au début (circulaire)
        ptr = (ptr + batch_size) % self.memory_size
        self.ptr[0] = ptr

    def forward(self):
        # Retourne toute la banque de mémoire actuelle
        return self.memory

# Initialiser la banque de mémoire M (par exemple 4096 vecteurs de 128 dimensions chacun)
memory_bank = MemoryBank(feature_dim=128, memory_size=4096).to(device)

print(f"Banque de mémoire M initialisée avec succès !")
print(f"Dimension de la banque M (nombre d'échantillons, nombre de dimensions) : {memory_bank.memory.shape}")

Banque de mémoire M initialisée avec succès !
Dimension de la banque M (nombre d'échantillons, nombre de dimensions) : torch.Size([4096, 128])


In [18]:
import torch.optim as optim

def contrastive_loss(z1, z2, memory, temperature=0.07):
    """
    Fonction de calcul de la perte InfoNCE à partir de la banque de mémoire M
    """
    # z1 et z2 ont pour dimension [Batch_size, 128]
    # Calculer la similarité (cosine similarity) entre z1 et z2 (paire positive)
    l_pos = torch.sum(z1 * z2, dim=-1, keepdim=True) # [B, 1]
    
    # Calculer la similarité entre z1 et toute la banque de mémoire M (paires négatives)
    # memory a pour dimension [Memory_size, 128]
    l_neg = torch.mm(z1, memory.t()) # [B, Memory_size]
    
    # Concaténer en logits et diviser par la température (temperature scaling)
    logits = torch.cat([l_pos, l_neg], dim=1) / temperature
    
    # Étiquette cible : z1 correspond à la première position positive (index 0)
    labels = torch.zeros(logits.shape[0], dtype=torch.long, device=logits.device)
    
    return F.cross_entropy(logits, labels)

# Initialiser l'optimiseur (algorithme d'optimisation Adam pour f)
optimizer = optim.Adam(f_encoder.parameters(), lr=1e-3)

# Entraînement d'essai sur quelques epochs (par exemple 5 epochs pour vérifier que le modèle tourne correctement)
num_epochs = 5
print("Début du pré-entraînement auto-supervisé...")

for epoch in range(num_epochs):
    f_encoder.train()
    total_loss = 0.0
    
    for batch_idx, (images, _) in enumerate(unlabeled_loader):
        images = images.to(device)
        
        # 1. Créer 2 vues aléatoires de l'image (augmentation simple : ajout d'un peu de bruit ou légère transformation)
        # Ici, x1 et x2 sont créés en ajoutant un léger bruit gaussien pour différencier les vues
        x1 = images + 0.1 * torch.randn_like(images)
        x2 = images + 0.1 * torch.randn_like(images)
        
        # Borner les valeurs à l'intervalle normalisé [-1, 1]
        x1 = torch.clamp(x1, -1.0, 1.0)
        x2 = torch.clamp(x2, -1.0, 1.0)
        
        # 2. Passage avant (forward) dans les 2 réseaux
        z1 = f_encoder(x1) # Online encoder
        
        with torch.no_grad():
            g_encoder.eval()
            z2 = g_encoder(x2) # Momentum encoder
            
        # 3. Calcul de la perte (Loss)
        loss = contrastive_loss(z1, z2, memory_bank())
        
        # 4. Rétropropagation et mise à jour de f
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # 5. Mettre à jour g via EMA et mettre à jour la banque de mémoire M
        update_momentum_encoder(f_encoder, g_encoder, m=0.999)
        memory_bank.update(z2.detach())
        
        total_loss += loss.item()
        
    avg_loss = total_loss / len(unlabeled_loader)
    print(f"Epoch [{epoch+1}/{num_epochs}] - Perte auto-supervisée : {avg_loss:.4f}")

print("Pré-entraînement auto-supervisé terminé !")

Début du pré-entraînement auto-supervisé...
Epoch [1/5] - Perte auto-supervisée : 4.0424
Epoch [2/5] - Perte auto-supervisée : 0.6615
Epoch [3/5] - Perte auto-supervisée : 0.1065
Epoch [4/5] - Perte auto-supervisée : 0.0520
Epoch [5/5] - Perte auto-supervisée : 0.0392
Pré-entraînement auto-supervisé terminé !


In [2]:
import torch.nn as nn
import torch.optim as optim

# 1. Geler complètement l'encodeur f (ses poids ne sont plus mis à jour)
for param in f_encoder.parameters():
    param.requires_grad = False

# 2. Construire un classifieur linéaire simple (entrée de dimension 128 issue de f, sortie de 10 classes : chiffres 0-9)
class LinearClassifier(nn.Module):
    def __init__(self, feature_dim=128, num_classes=10):
        super().__init__()
        self.fc = nn.Linear(feature_dim, num_classes)
        
    def forward(self, z):
        return self.fc(z)

classifier = LinearClassifier(feature_dim=128, num_classes=10).to(device)

# 3. Configurer l'optimiseur uniquement pour le classifieur linéaire, ainsi que la fonction de perte
optimizer_cls = optim.Adam(classifier.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

# 4. Entraîner le classifieur linéaire avec exactement les 100 échantillons étiquetés (labeled_loader)
num_finetune_epochs = 30
f_encoder.eval() # Mettre f en mode inférence (extraction des caractéristiques uniquement)

print("Début du fine-tuning du classifieur linéaire avec 100 échantillons étiquetés...")
for epoch in range(num_finetune_epochs):
    classifier.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for images, labels in labeled_loader:
        images, labels = images.to(device), labels.to(device)
        
        with torch.no_grad():
            features = f_encoder(images) # Récupérer le vecteur de dimension 128 issu de f pré-entraîné
            
        outputs = classifier(features)
        loss = criterion(outputs, labels)
        
        optimizer_cls.zero_grad()
        loss.backward()
        optimizer_cls.step()
        
        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
    if (epoch + 1) % 10 == 0 or epoch == 0:
        acc = 100. * correct / total
        print(f"Epoch [{epoch+1}/{num_finetune_epochs}] - Perte : {total_loss/len(labeled_loader):.4f} - Préc. sur les 100 échantillons d'entraînement : {acc:.2f}%")

# 5. Évaluer le modèle complet sur l'ensemble de test standard (10 000 images)
classifier.eval()
test_correct = 0
test_total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        features = f_encoder(images)
        outputs = classifier(features)
        _, predicted = outputs.max(1)
        test_total += labels.size(0)
        test_correct += predicted.eq(labels).sum().item()

test_acc = 100. * test_correct / test_total
print("\n" + "="*40)
print( "résultat final de l'évaluation" )
print(f"PRÉCISION SUR L'ENSEMBLE DE TEST : {test_acc:.2f}%")
print("="*40)

NameError: name 'f_encoder' is not defined